In [1]:
import json
from time import time

import pandas as pd
from kafka import KafkaProducer

In [2]:
file = "data/green_tripdata_2025-10.parquet"

columns = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "tip_amount",
    "total_amount"
]

df = pd.read_parquet(file, columns=columns)

df["lpep_pickup_datetime"] = df["lpep_pickup_datetime"].astype(str)
df["lpep_dropoff_datetime"] = df["lpep_dropoff_datetime"].astype(str)

df = df.astype(object).where(pd.notnull(df), None)

print("Rows loaded:", len(df))
print(df[df["passenger_count"].isna()].head())
print(df.head())

Rows loaded: 49416
      lpep_pickup_datetime lpep_dropoff_datetime PULocationID DOLocationID  \
44401  2025-10-01 04:30:14   2025-10-01 05:16:45          119           61   
44402  2025-10-01 04:36:31   2025-10-01 04:57:37           69          107   
44403  2025-10-01 04:26:47   2025-10-01 05:00:46           71          137   
44404  2025-10-01 04:43:06   2025-10-01 05:00:16           17          137   
44405  2025-10-01 05:57:21   2025-10-01 07:05:43           10          146   

      passenger_count trip_distance tip_amount total_amount  
44401            None         14.77        0.0        46.73  
44402            None          8.62        0.0        29.66  
44403            None         11.83        0.0        41.38  
44404            None          5.94        0.0        27.68  
44405            None         12.43        0.0        37.43  
  lpep_pickup_datetime lpep_dropoff_datetime PULocationID DOLocationID  \
0  2025-10-01 00:21:47   2025-10-01 00:24:37          247         

In [3]:
df["lpep_pickup_datetime"] = df["lpep_pickup_datetime"].astype(str)
df["lpep_dropoff_datetime"] = df["lpep_dropoff_datetime"].astype(str)

In [4]:
TOPIC = "green-trips"
BOOTSTRAP_SERVERS = "localhost:9092"

producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

In [8]:
from time import time

t0 = time()

for row in df.to_dict(orient="records"):
    producer.send(TOPIC, value=row)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')

took 2.77 seconds


In [6]:
records = df.to_dict(orient="records")
print(records[0])

bad = [r for r in records if str(r.get("passenger_count")) == "nan"]
print("bad passenger_count rows:", len(bad))

{'lpep_pickup_datetime': '2025-10-01 00:21:47', 'lpep_dropoff_datetime': '2025-10-01 00:24:37', 'PULocationID': 247, 'DOLocationID': 69, 'passenger_count': 1.0, 'trip_distance': 0.7, 'tip_amount': 1.7, 'total_amount': 10.0}
bad passenger_count rows: 0
